手写LORA


In [1]:
import transformers
import peft
import datasets
import tqdm
import accelerate
import einops
import torch
from torch import nn
import torch.nn.functional as F
import copy
from typing import List

In [2]:
class MyLora(nn.Module):
    def __init__(self,base_layer: nn.Linear,alpha:int = 1, lora_rank:int=8, dropout_p:float=0.0, test_mode: bool = False):
        
        super(MyLora,self).__init__()
        self.base_layer = copy.deepcopy(base_layer)
        self.lora_rank = lora_rank
        self.alpha = alpha
        self.test_mode = test_mode
        self.A = nn.Parameter(torch.empty((lora_rank, base_layer.in_features),dtype=base_layer.weight.dtype))
        self.B = nn.Parameter(torch.empty((base_layer.out_features, lora_rank),dtype=base_layer.weight.dtype))
        self.dropout = nn.Dropout(dropout_p)
        # 初始化 lora 矩阵
        nn.init.normal_(self.A, mean=0.0, std=0.02)
        if test_mode:
            nn.init.normal_(self.B, mean=0.0, std=0.02)
        else:
            nn.init.zeros_(self.B)

        # 冻结原来的层的参数
        for param in self.base_layer.parameters():
            param.requires_grad = False

    def forward(self, x):
        scaling = float(self.alpha) / float(self.lora_rank)
        h = F.linear(self.dropout(x),self.A)
        output = F.linear(h,self.B)
        return self.base_layer(x) + output * scaling

替换

In [3]:
def replace_lora(model, lora_rank=8, alpha=1, dropout_p=0.1, test_mode=False, 
    embed_requires_grad: bool = False,      # embedding 层是否训练
    norm_requires_grad: bool = False,       # norm 层是否训练
    head_requires_grad: bool = False,       # lm_head 层是否训练（Causal LM才有）
    ):
    for name, module in model.named_children():
        if any(s in name for s in ['embed', 'norm', 'lm_head']):
            requires_grad = embed_requires_grad if 'embed' in name \
                            else norm_requires_grad if 'norm' in name \
                            else head_requires_grad
            for param in module.parameters():
                param.requires_grad = requires_grad
        elif isinstance(module, nn.Linear) and module.weight.requires_grad:
            new_module = MyLora(module, lora_rank=lora_rank, alpha=alpha, dropout_p=dropout_p, test_mode=test_mode)
            setattr(model, name, new_module)
        else:
            replace_lora(module, lora_rank=lora_rank, alpha=alpha, dropout_p=dropout_p, test_mode=test_mode,embed_requires_grad=embed_requires_grad,
                norm_requires_grad=norm_requires_grad, head_requires_grad=head_requires_grad)

In [4]:
def print_trainable_parameters(model:nn.Module):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    percent = trainable_params / total_params * 100 if total_params > 0 else 0
    print(f"Total parameters: {total_params}, Trainable parameters: {trainable_params}, "
          f"Trainable percentage: {percent:.2f}%")

验证一下

In [5]:
from transformers import AutoConfig

config = AutoConfig.for_model('llama')
config.hidden_size = 24
config.intermediate_size = config.hidden_size * 4
config.num_attention_heads = 4
config.num_hidden_layers = 4
config.num_key_value_heads = 2
config.vocab_size = 128

In [6]:
from transformers import AutoModel, AutoModelForCausalLM

raw_model = AutoModel.from_config(config)  # 没带因果头
# raw_model = AutoModelForCausalLM.from_config(config)  # 带了因果头
print(raw_model)

"""
LlamaModel(
  (embed_tokens): Embedding(128, 24)
  (layers): ModuleList(
    (0-3): 4 x LlamaDecoderLayer(
      (self_attn): LlamaSdpaAttention(
        (q_proj): Linear(in_features=24, out_features=24, bias=False)
        (k_proj): Linear(in_features=24, out_features=12, bias=False)
        (v_proj): Linear(in_features=24, out_features=12, bias=False)
        (o_proj): Linear(in_features=24, out_features=24, bias=False)
        (rotary_emb): LlamaRotaryEmbedding()
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=24, out_features=96, bias=False)
        (up_proj): Linear(in_features=24, out_features=96, bias=False)
        (down_proj): Linear(in_features=96, out_features=24, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm()
      (post_attention_layernorm): LlamaRMSNorm()
    )
  )
  (norm): LlamaRMSNorm()
)
"""

LlamaModel(
  (embed_tokens): Embedding(128, 24)
  (layers): ModuleList(
    (0-3): 4 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=24, out_features=512, bias=False)
        (k_proj): Linear(in_features=24, out_features=256, bias=False)
        (v_proj): Linear(in_features=24, out_features=256, bias=False)
        (o_proj): Linear(in_features=512, out_features=24, bias=False)
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=24, out_features=96, bias=False)
        (up_proj): Linear(in_features=24, out_features=96, bias=False)
        (down_proj): Linear(in_features=96, out_features=24, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm((24,), eps=1e-06)
      (post_attention_layernorm): LlamaRMSNorm((24,), eps=1e-06)
    )
  )
  (norm): LlamaRMSNorm((24,), eps=1e-06)
  (rotary_emb): LlamaRotaryEmbedding()
)


'\nLlamaModel(\n  (embed_tokens): Embedding(128, 24)\n  (layers): ModuleList(\n    (0-3): 4 x LlamaDecoderLayer(\n      (self_attn): LlamaSdpaAttention(\n        (q_proj): Linear(in_features=24, out_features=24, bias=False)\n        (k_proj): Linear(in_features=24, out_features=12, bias=False)\n        (v_proj): Linear(in_features=24, out_features=12, bias=False)\n        (o_proj): Linear(in_features=24, out_features=24, bias=False)\n        (rotary_emb): LlamaRotaryEmbedding()\n      )\n      (mlp): LlamaMLP(\n        (gate_proj): Linear(in_features=24, out_features=96, bias=False)\n        (up_proj): Linear(in_features=24, out_features=96, bias=False)\n        (down_proj): Linear(in_features=96, out_features=24, bias=False)\n        (act_fn): SiLU()\n      )\n      (input_layernorm): LlamaRMSNorm()\n      (post_attention_layernorm): LlamaRMSNorm()\n    )\n  )\n  (norm): LlamaRMSNorm()\n)\n'

In [7]:
print_trainable_parameters(raw_model)

Total parameters: 178392, Trainable parameters: 178392, Trainable percentage: 100.00%


In [8]:
lora_model =  copy.deepcopy(raw_model)  # 深克隆，独立一个新模型
replace_lora(lora_model, lora_rank=8, alpha=16)  # 替换
print_trainable_parameters(lora_model) # 打印参数情况
print(lora_model)

"""
trainable params: 16,896 || all params: 54,744 || trainable%: 30.8637

LlamaModel(
  (embed_tokens): Embedding(128, 24)
  (layers): ModuleList(
    (0-3): 4 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): LoraLinear(
          (base_layer): Linear(in_features=24, out_features=24, bias=False)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (k_proj): LoraLinear(
          (base_layer): Linear(in_features=24, out_features=12, bias=False)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (v_proj): LoraLinear(
          (base_layer): Linear(in_features=24, out_features=12, bias=False)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (o_proj): LoraLinear(
          (base_layer): Linear(in_features=24, out_features=24, bias=False)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (rotary_emb): LlamaRotaryEmbedding()
      )
      (mlp): LlamaMLP(
        (gate_proj): LoraLinear(
          (base_layer): Linear(in_features=24, out_features=96, bias=False)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (up_proj): LoraLinear(
          (base_layer): Linear(in_features=24, out_features=96, bias=False)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (down_proj): LoraLinear(
          (base_layer): Linear(in_features=96, out_features=24, bias=False)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm()
      (post_attention_layernorm): LlamaRMSNorm()
    )
  )
  (norm): LlamaRMSNorm()
)
"""

Total parameters: 242136, Trainable parameters: 63744, Trainable percentage: 26.33%
LlamaModel(
  (embed_tokens): Embedding(128, 24)
  (layers): ModuleList(
    (0-3): 4 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): MyLora(
          (base_layer): Linear(in_features=24, out_features=512, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (k_proj): MyLora(
          (base_layer): Linear(in_features=24, out_features=256, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (v_proj): MyLora(
          (base_layer): Linear(in_features=24, out_features=256, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (o_proj): MyLora(
          (base_layer): Linear(in_features=512, out_features=24, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (mlp): LlamaMLP(
        (gate_proj): MyLora(
          (base_layer): Linear(in_features=24, out_feature

'\ntrainable params: 16,896 || all params: 54,744 || trainable%: 30.8637\n\nLlamaModel(\n  (embed_tokens): Embedding(128, 24)\n  (layers): ModuleList(\n    (0-3): 4 x LlamaDecoderLayer(\n      (self_attn): LlamaAttention(\n        (q_proj): LoraLinear(\n          (base_layer): Linear(in_features=24, out_features=24, bias=False)\n          (dropout): Dropout(p=0.0, inplace=False)\n        )\n        (k_proj): LoraLinear(\n          (base_layer): Linear(in_features=24, out_features=12, bias=False)\n          (dropout): Dropout(p=0.0, inplace=False)\n        )\n        (v_proj): LoraLinear(\n          (base_layer): Linear(in_features=24, out_features=12, bias=False)\n          (dropout): Dropout(p=0.0, inplace=False)\n        )\n        (o_proj): LoraLinear(\n          (base_layer): Linear(in_features=24, out_features=24, bias=False)\n          (dropout): Dropout(p=0.0, inplace=False)\n        )\n        (rotary_emb): LlamaRotaryEmbedding()\n      )\n      (mlp): LlamaMLP(\n        (gate_

查看是不是只有Lora层是可训练

In [9]:
def print_model_parameters(model):
    """
    查看模型参数的 requires_grad 情况
    """
    print("Layer Name & Parameters")
    print("----------------------------")
    for name, parameter in model.named_parameters():
        print(f"{name:50} | Requires_grad: {parameter.requires_grad}")

In [10]:
print_model_parameters(lora_model)

Layer Name & Parameters
----------------------------
embed_tokens.weight                                | Requires_grad: False
layers.0.self_attn.q_proj.A                        | Requires_grad: True
layers.0.self_attn.q_proj.B                        | Requires_grad: True
layers.0.self_attn.q_proj.base_layer.weight        | Requires_grad: False
layers.0.self_attn.k_proj.A                        | Requires_grad: True
layers.0.self_attn.k_proj.B                        | Requires_grad: True
layers.0.self_attn.k_proj.base_layer.weight        | Requires_grad: False
layers.0.self_attn.v_proj.A                        | Requires_grad: True
layers.0.self_attn.v_proj.B                        | Requires_grad: True
layers.0.self_attn.v_proj.base_layer.weight        | Requires_grad: False
layers.0.self_attn.o_proj.A                        | Requires_grad: True
layers.0.self_attn.o_proj.B                        | Requires_grad: True
layers.0.self_attn.o_proj.base_layer.weight        | Requires_grad:

In [11]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules='all-linear', # 太低版本的 peft 不支持这种做法
)
peft_lora_model = copy.deepcopy(raw_model)
peft_lora_model = get_peft_model(peft_lora_model, lora_config)
peft_lora_model.print_trainable_parameters()

"""
trainable params: 16,896 || all params: 54,744 || trainable%: 30.8637
"""

trainable params: 63,744 || all params: 242,136 || trainable%: 26.3257


'\ntrainable params: 16,896 || all params: 54,744 || trainable%: 30.8637\n'

卸载和重载

In [12]:
def unload_lora(model: nn.Module,adapter_name: str = "adapter"):
    """
    卸载 LoRA 模块
    """
    modules_to_save = {}
    def search_lora_linear(model: nn.Module, prefix: List[str]):
        for name, module in model.named_children():
            new_prefix = prefix + [name]
            if isinstance(module, MyLora):
                # 替换为原始的 Linear 层
                modules_to_save['.'.join(new_prefix)] = {
                    "lora_A_weight": module.A.data.cpu(),
                    "lora_B_weight": module.B.data.cpu(),
                    "r": module.lora_rank,
                    "alpha": module.alpha,
                    "dropout_p": module.dropout.p,
                }
                setattr(model, name, module.base_layer)
            else:
                search_lora_linear(module, prefix=new_prefix)

    search_lora_linear(model, [])
    # 解冻原模型
    for name, param in model.named_parameters():
        param.requires_grad = True
    torch.save(modules_to_save, f"{adapter_name}.pt")

In [13]:
def load_lora(model: nn.Module,adapter_name: str = "adapter"):
    """
    卸载 LoRA 模块
    """
    lora_parameters = torch.load(f"{adapter_name}.pt")
    for name,lora_params in lora_parameters.items():
        child = dict(model.named_modules())[name]
        if isinstance(child, nn.Linear):
            if name in lora_parameters:            
                lora_linear = MyLora(child, lora_params['r'], lora_params['alpha'], lora_params['dropout_p'])
                lora_linear.A.data = lora_params["lora_A_weight"].to(lora_linear.A.device)
                lora_linear.B.data = lora_params["lora_B_weight"].to(lora_linear.B.device)

                parts = name.split(".")
                obj = model            
                for part in parts[:-1]:  # 不包括最后一级
                    obj = getattr(obj, part)

                setattr(obj, parts[-1], lora_linear)
    # 冻结原模型
    for name, param in model.named_parameters():
        if any(s in name for s in ['embed', 'norm', 'lm_head']):
            param.requires_grad = False


In [14]:
torch.__version__

'2.1.1+cu118'

Test

In [15]:
# 创建一个测试 tensor
bsz = 2
seq_len = 8
test_tensor = torch.randint(0, config.vocab_size, (bsz, seq_len))

In [16]:
# 开测试模式，让 BA 非零
lora_model = copy.deepcopy(raw_model)
replace_lora(lora_model, lora_rank=8, alpha=16, test_mode=True)

In [17]:
# 原模型的前向结果
raw_model.eval()
print_trainable_parameters(raw_model)   # 检查参数和可训练情况
raw_res = raw_model(test_tensor).last_hidden_state
"""
trainable params: 37,848 || all params: 37,848 || trainable%: 100.0000
"""

# 第一次直接初始化 lora 的前向结果
lora_model.eval()
print_trainable_parameters(lora_model)  # 检查参数和可训练情况
before_unload_res = lora_model(test_tensor).last_hidden_state
"""
trainable params: 16,896 || all params: 54,744 || trainable%: 30.8637
"""

# 卸载 lora 后的前向结果
unload_lora(lora_model)
lora_model.eval()
print_trainable_parameters(lora_model)  # 检查参数和可训练情况
unload_res = lora_model(test_tensor).last_hidden_state
"""
trainable params: 37,848 || all params: 37,848 || trainable%: 100.0000
"""

# 重新装载 lora 后的前向结果
load_lora(lora_model)
lora_model.eval()
print_trainable_parameters(lora_model)  # 检查参数和可训练情况
load_res = lora_model(test_tensor).last_hidden_state
"""
trainable params: 16,896 || all params: 54,744 || trainable%: 30.8637
"""

Total parameters: 178392, Trainable parameters: 178392, Trainable percentage: 100.00%
Total parameters: 242136, Trainable parameters: 63744, Trainable percentage: 26.33%
Total parameters: 178392, Trainable parameters: 178392, Trainable percentage: 100.00%
Total parameters: 242136, Trainable parameters: 63744, Trainable percentage: 26.33%


'\ntrainable params: 16,896 || all params: 54,744 || trainable%: 30.8637\n'

训练测试

In [18]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from tqdm import tqdm
from typing import List
from einops import rearrange
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import AutoConfig, AutoTokenizer, AutoModel, AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if device != 'cpu' and torch.cuda.is_bf16_supported() else torch.float32
print(f'device: {device}\ndtype: {dtype}')

device: cuda
dtype: torch.bfloat16


In [19]:
model_name_or_path = 'Qwen/Qwen1.5-0.5B'

# 加载原始模型
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(model_name_or_path, torch_dtype=dtype).to(device)

In [20]:
replace_lora(model, lora_rank=16, alpha=32, dropout_p=0.0)
model.to(device)

# 查看可训练参数
print_trainable_parameters(model)
"""
trainable params: 3,784,704 || all params: 467,772,416 || trainable%: 0.8091
"""

Total parameters: 471557120, Trainable parameters: 7569408, Trainable percentage: 1.61%


'\ntrainable params: 3,784,704 || all params: 467,772,416 || trainable%: 0.8091\n'

In [21]:
# 数据路径可以改成本地
data_name_or_path = 'bio-nlp-umass/bioinstruct'

# 定义训练数据集
class SFTDataset(Dataset):
    def __init__(self,
        tokenizer: AutoTokenizer,
        data_path: str,
        load_local: bool = False,
        max_len: int = 256,
        split_len: str = '1%',
    ):
        super().__init__()
        self.tokenizer = tokenizer

        if load_local:
            ds = load_dataset('json', data_dir=data_path, split=f'train[:{split_len}]')
        else:
            ds = load_dataset(data_path, split=f'train[:{split_len}]')
        self.max_len = max_len

        def process_func(example):
            # 提取 instruction 和 input
            instruction = example['instruction'].strip()
            input = example['input'].strip()
            output = example['output'].strip()

            # 构造模板
            instruction_msg = [
                {"role": "user", "content": (instruction + f"\n{input}") if len(input) > 0 else instruction}
            ]
            tokenized_instruction = tokenizer.apply_chat_template(instruction_msg, tokenize=True, add_generation_prompt=True)
            tokenized_output = tokenizer(output + "<|im_end|>" + f"{tokenizer.eos_token}\n")['input_ids']

            # 截断，最大不超过 max_len
            tokenized_prompt = (tokenized_instruction + tokenized_output)[:self.max_len]

            # 构造 input_ids, attention_mask, labels
            input_ids = tokenized_prompt[:-1]
            padding_mask = ([0] * len(tokenized_instruction) + [1] * (len(tokenized_output)))[:self.max_len][1:]
            labels = tokenized_prompt[1:]

            return {
                'input_ids': input_ids,
                'attention_mask': padding_mask,
                'labels': labels,
            }

        self.ds = ds.map(
            process_func,
            batched=False,
            remove_columns=ds.column_names,
            desc='Processing dataset',
        )

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, index: int):
        return self.ds[index]

In [22]:
ds = SFTDataset(tokenizer, data_name_or_path, load_local=False, split_len="1%")

print(len(ds[0]['input_ids']))
print(len(ds[0]['attention_mask']))
print(len(ds[0]['labels']))

print(tokenizer.decode(ds[0]['input_ids']))
print(ds[0]['attention_mask'])
print(tokenizer.decode(ds[0]['labels']))

"""
79
79
79

<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Identify the main conclusion from the provided medical report excerpt.
The patient's blood test results showed an elevation in liver enzymes, specifically ALT and AST, which suggests potential liver damage. Additionally, the patient's ultrasound showed a fatty liver.<|im_end|>
<|im_start|>assistant
The patient has signs of liver damage and a fatty liver.<|im_end|><|endoftext|>

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

system
You are a helpful assistant<|im_end|>
<|im_start|>user
Identify the main conclusion from the provided medical report excerpt.
The patient's blood test results showed an elevation in liver enzymes, specifically ALT and AST, which suggests potential liver damage. Additionally, the patient's ultrasound showed a fatty liver.<|im_end|>
<|im_start|>assistant
The patient has signs of liver damage and a fatty liver.<|im_end|><|endoftext|>
"""

README.md: 0.00B [00:00, ?B/s]

d:\ProgramData\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\41127\.cache\huggingface\hub\datasets--bio-nlp-umass--bioinstruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


biomed_instruct_25k.json:   0%|          | 0.00/13.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25005 [00:00<?, ? examples/s]

Processing dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

79
79
79
<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Identify the main conclusion from the provided medical report excerpt.
The patient's blood test results showed an elevation in liver enzymes, specifically ALT and AST, which suggests potential liver damage. Additionally, the patient's ultrasound showed a fatty liver.<|im_end|>
<|im_start|>assistant
The patient has signs of liver damage and a fatty liver.<|im_end|><|endoftext|>
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
system
You are a helpful assistant<|im_end|>
<|im_start|>user
Identify the main conclusion from the provided medical report excerpt.
The patient's blood test results showed an elevation in liver enzymes, specifically ALT and AST, which suggests potential liver damage. Additionally, the patient's ul

"\n79\n79\n79\n\n<|im_start|>system\nYou are a helpful assistant<|im_end|>\n<|im_start|>user\nIdentify the main conclusion from the provided medical report excerpt.\nThe patient's blood test results showed an elevation in liver enzymes, specifically ALT and AST, which suggests potential liver damage. Additionally, the patient's ultrasound showed a fatty liver.<|im_end|>\n<|im_start|>assistant\nThe patient has signs of liver damage and a fatty liver.<|im_end|><|endoftext|>\n\n[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]\n\nsystem\nYou are a helpful assistant<|im_end|>\n<|im_start|>user\nIdentify the main conclusion from the provided medical report excerpt.\nThe patient's blood test results showed an elevation in liver enzymes, specifically ALT and AST, which suggests potential liver damage. Additi

In [23]:
def collate_fn(batch: List, tokenizer):
    max_len = max(len(item['input_ids']) for item in batch)

    input_ids = []
    attention_mask = []
    labels = []

    for item in batch:
        input_id = item['input_ids']
        attention_mask_item = item['attention_mask']
        label = item['labels']

        # 计算填充长度
        pad_len = max_len - len(input_id)

        # 左填充
        input_ids.append([tokenizer.eos_token_id] * pad_len + input_id)
        attention_mask.append([0] * pad_len + attention_mask_item)
        labels.append([tokenizer.eos_token_id] * pad_len + label)

    # 将 list 转换为 tensor
    input_ids = torch.LongTensor(input_ids)
    attention_mask = torch.LongTensor(attention_mask)
    labels = torch.LongTensor(labels)

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels,
    }

In [26]:
bsz = 8

dataloader = DataLoader(ds, batch_size=bsz, shuffle=True, collate_fn=lambda batch: collate_fn(batch, tokenizer))

for batch in dataloader:
    print(batch)
    break

# 这里的三个张量形状应该一致，都为 [bsz, seq_len]
"""
{
  "input_ids": ...,
  "attention_mask": ...,
  "labels": ...
}
"""

{'input_ids': tensor([[151644,   8948,    198,  ...,   1378,  32019,  41711],
        [151643, 151643, 151643,  ...,     13, 151645, 151643],
        [151643, 151643, 151643,  ...,   3007, 151645, 151643],
        ...,
        [151643, 151643, 151643,  ...,     13, 151645, 151643],
        [151643, 151643, 151643,  ...,     13, 151645, 151643],
        [151643, 151643, 151643,  ...,   7786, 151645, 151643]]), 'attention_mask': tensor([[0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        ...,
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1]]), 'labels': tensor([[  8948,    198,   2610,  ...,  32019,  41711,  10975],
        [151643, 151643, 151643,  ..., 151645, 151643,    198],
        [151643, 151643, 151643,  ..., 151645, 151643,    198],
        ...,
        [151643, 151643, 151643,  ..., 151645, 151643,    198],
        [151643, 151643, 151643,  ..., 151645, 151643,    198],
       

'\n{\n  "input_ids": ...,\n  "attention_mask": ...,\n  "labels": ...\n}\n'

In [24]:
lr = 5e-4       # 学习率，不要太大
num_epochs = 3    # 训练轮数
logging_steps = 5   # 每隔多少步输出
max_grad_norm = 1.0  # 最大梯度范数，用于剪裁

optimizer = optim.AdamW(model.parameters(), lr=lr) # AdamW 优化器，常用

In [27]:
model.train()

total_loss = 0
total_step = 0
for epoch in range(num_epochs):
    for step, batch in enumerate(tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        
        # 重整 logits, mask, labels
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        logits = outputs.logits
        rearranged_logits = rearrange(logits, 'bsz seq_len vocab_size -> (bsz seq_len) vocab_size')
        rearranged_attention_mask = rearrange(attention_mask, 'bsz seq_len -> (bsz seq_len)')
        rearranged_labels = rearrange(labels, 'bsz seq_len -> (bsz seq_len)')
    
        # 按照 mask 手动计算 loss
        sum_loss = F.cross_entropy(rearranged_logits, rearranged_labels, ignore_index=0, reduction='none')
        loss = torch.sum(sum_loss * rearranged_attention_mask) / torch.sum(rearranged_attention_mask)
        loss.backward()

        # 计算梯度范数并裁剪
        total_norm = nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

        optimizer.step()
        total_loss += loss.item()

        total_step += 1
        if total_step % logging_steps == 0:
            avg_loss = total_loss / total_step
            print(f"Step: {step+1}/{len(dataloader)}, Loss: {avg_loss:.4f}, Grad Norm: {total_norm:.4f}", flush=True)


    # 打印每个 epoch 结束的累计损失
    print(f"Epoch {epoch+1} finished, Average Loss: {total_loss / total_step:.4f}", flush=True)

Epoch 1/3:  12%|█▎        | 4/32 [00:00<00:06,  4.62it/s]

Step: 5/32, Loss: 2.1969, Grad Norm: 2.9844


Epoch 1/3:  28%|██▊       | 9/32 [00:01<00:03,  7.48it/s]

Step: 10/32, Loss: 2.0734, Grad Norm: 2.3125


Epoch 1/3:  41%|████      | 13/32 [00:02<00:02,  7.86it/s]

...

Step: 26/32, Loss: 1.2515, Grad Norm: 1.9375


Epoch 3/3:  91%|█████████ | 29/32 [00:02<00:00, 10.49it/s]

Step: 31/32, Loss: 1.2154, Grad Norm: 2.0156


Epoch 3/3: 100%|██████████| 32/32 [00:03<00:00, 10.25it/s]

Epoch 3 finished, Average Loss: 1.2091


In [28]:
# 创建测试 tensor
test_text = 'Hello, world!'
test_tensor = tokenizer(test_text, return_tensors='pt').to(device)
test_tensor['input_ids'].shape
inputs = tokenizer(test_text)

# 手动移位，对应我们上面的实现
input_ids = torch.LongTensor([inputs['input_ids'][:-1]]).to(device)
attention_mask = torch.LongTensor([inputs['attention_mask'][:-1]]).to(device)
labels = torch.LongTensor([inputs['input_ids'][1:]]).to(device)

# 前向
outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)
loss, logits = outputs.loss, outputs.logits
loss.item()
"""
9.069315910339355
"""

'\n9.069315910339355\n'

In [29]:
# 用 einops 重整 logits、mask、labels，和上面一样
rearranged_logits = rearrange(logits, 'b seq_len vocab_size -> (b seq_len) vocab_size')
rearranged_attention_mask = rearrange(attention_mask, 'b seq_len -> (b seq_len)')
rearranged_labels = rearrange(labels, 'b seq_len -> (b seq_len)')

# 手动计算 loss
sum_loss = F.cross_entropy(rearranged_logits, rearranged_labels, ignore_index=0, reduction='none')
print(sum_loss)
# 按 mask 计算最后的 loss，当然这里没有被 mask 的区域，相当于加和平均
torch.sum(sum_loss * rearranged_attention_mask) / torch.sum(rearranged_attention_mask)
"""
tensor([14.0533,  1.8505, 10.2932,  3.9565], grad_fn=<NllLossBackward0>)
tensor(7.5384, grad_fn=<DivBackward0>)
"""

tensor([8.8125, 7.7812, 0.0000], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<NllLossBackward0>)


'\ntensor([14.0533,  1.8505, 10.2932,  3.9565], grad_fn=<NllLossBackward0>)\ntensor(7.5384, grad_fn=<DivBackward0>)\n'

In [30]:
inputs = tokenizer(test_text)

# 这次不移位
input_ids = torch.LongTensor([inputs['input_ids']]).to(device)
attention_mask = torch.LongTensor([inputs['attention_mask']]).to(device)
labels = torch.LongTensor([inputs['input_ids']]).to(device)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)
loss, logits = outputs.loss, outputs.logits
loss
"""
tensor(7.5384, grad_fn=<NllLossBackward0>)
"""

'\ntensor(7.5384, grad_fn=<NllLossBackward0>)\n'

In [31]:
def inference(
    model,
    tokenizer,
    text: str,
    max_new_tokens: int = 160,
    do_sample: bool = True,
    temperature: float = 0.3,
    print_inputs: bool = True,
    streaming: bool = False,
):
    # 构建输入，模板要和 Dataset 中一致
    prompt_msg = [
        {"role": "user", "content": text}
    ]
    prompt = tokenizer.apply_chat_template(prompt_msg, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(device)
    input_ids = inputs['input_ids']
    im_end_id = tokenizer.encode("<|im_end|>")[0]

    # 是否打印输入部分
    if print_inputs: 
        print(prompt, end='')
    
    # 生成
    stop_words = [tokenizer.eos_token_id, im_end_id]
    generated_tokens = []

    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids)
        
        logits = outputs.logits[:, -1, :]
        
        # 不同采样方式
        if do_sample:
            logits = logits / temperature
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        else:
            # 贪婪解码
            next_token = torch.argmax(logits, dim=-1, keepdim=True)
        if next_token.item() in stop_words:
            break
        generated_tokens.append(next_token.item())
        # 流式输出
        if streaming:
            yield tokenizer.decode(generated_tokens)
        
        # 更新输入
        input_ids = torch.cat([input_ids, next_token], dim=-1)
    
    generated_text = tokenizer.decode(generated_tokens)
    return generated_text

In [32]:
model.eval()

for test_text in [
    'Describe the process of bacterial conjugation and its significance in the context of antibiotic resistance.',
    'Explain the role of insulin in the body and how insulin resistance affects blood sugar levels.',
    'Provide recommendations for lifestyle changes that can help improve the overall health of a patient with type 2 diabetes.',
]:
    print('=' * 80)
    last_text = ''
    for text in inference(model, tokenizer, test_text, streaming=True):
        cur_text = text.replace(last_text, '')
        print(cur_text, end='', flush=True)
        last_text = text
    print('\n')

"""
================================================================================
<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Describe the process of bacterial conjugation and its significance in the context of antibiotic resistance.<|im_end|>
<|im_start|>assistant
Bacterial conjugation is a process by which bacteria exchange genetic material through direct cell-to-cell contact. This process plays a crucial role in antibiotic resistance as it allows bacteria to inherit the genes of other bacteria, increasing their ability to resist certain antibiotics. Conjugation occurs when two bacteria exchange genetic material through a process involving cell相亲、细胞融合和细胞质融合, resulting in new bacterial species with enhanced antibiotic resistance genes. This process contributes to the spread of antibiotic resistance among bacterial populations and ultimately contributes to the global pandemic of antibiotic-resistant bacteria.

================================================================================
<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Explain the role of insulin in the body and how insulin resistance affects blood sugar levels.<|im_end|>
<|im_start|>assistant
Insulin is a hormone produced by the pancreas that helps regulate blood sugar levels. Its main function is to transport glucose (sugar) from the bloodstream to cells, where it is used for energy or stored. Insulin resistance occurs when the body's cells do not respond properly to insulin, leading to a decrease in insulin sensitivity and an increase in glucose production. This can result in higher blood sugar levels, which can lead to a range of health problems, including cardiovascular disease, diabetes, and some cancers. Insulin resistance can also contribute to the development of type 2 diabetes.

================================================================================
<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Provide recommendations for lifestyle changes that can help improve the overall health of a patient with type 2 diabetes.<|im_end|>
<|im_start|>assistant
1. Maintain a healthy diet: Eat a balanced diet with lean protein, whole grains, fruits, vegetables, and healthy fats. Limit consumption of sugary beverages, processed foods, and saturated fats.
2. Exercise regularly: Aim for at least 150 minutes of moderate-intensity exercise per week, such as walking, swimming, or biking. Consult with a healthcare professional to determine the appropriate exercise regimen.
3. Maintain a healthy weight: Focus on making healthy lifestyle choices and being mindful of portion sizes. Consult with a healthcare professional to determine the appropriate weight range for your patient.
4. Limit alcohol intake: Limit alcohol intake to no more than 150 ml per day, and consider limiting alcohol consumption while on medication.
5. Manage stress: Practice stress management techniques like deep breathing

"""

<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Describe the process of bacterial conjugation and its significance in the context of antibiotic resistance.<|im_end|>
<|im_start|>assistant
Bacterial conjugation is a process by which bacteria exchange genetic material through a process called conjugation. This allows bacteria to exchange genetic material, including genes, without the need for a plasmid. The process is essential for antibiotic resistance as it enables bacteria to exchange genetic material and develop resistance genes, which can then be transferred to other bacteria. This process helps bacteria develop a wide range of resistance genes, making them more susceptible to antibiotics and increasing their overall population.

<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Explain the role of insulin in the body and how insulin resistance affects blood sugar levels.<|im_end|>
<|im_start|>assistant
Insulin is a hormone that help

"\n================================================================================\n<|im_start|>system\nYou are a helpful assistant<|im_end|>\n<|im_start|>user\nDescribe the process of bacterial conjugation and its significance in the context of antibiotic resistance.<|im_end|>\n<|im_start|>assistant\nBacterial conjugation is a process by which bacteria exchange genetic material through direct cell-to-cell contact. This process plays a crucial role in antibiotic resistance as it allows bacteria to inherit the genes of other bacteria, increasing their ability to resist certain antibiotics. Conjugation occurs when two bacteria exchange genetic material through a process involving cell相亲、细胞融合和细胞质融合, resulting in new bacterial species with enhanced antibiotic resistance genes. This process contributes to the spread of antibiotic resistance among bacterial populations and ultimately contributes to the global pandemic of antibiotic-resistant bacteria.\n\n====================================

In [35]:
def merge_lora(module: nn.Module):
    """
    将 lora 参数合并到原来的 base_layer 中，并将 lora 层替换回原来的 nn.Linear 层
    """
    def search_lora_linear(module: nn.Module, prefix: List[str]):
        for name, child in module.named_children():
            new_prefix = prefix + [name]
            if isinstance(child, MyLora):
                # 合并 lora 参数到 base_layer
                with torch.no_grad():
                    lora_adjustment = torch.matmul(child.B, child.A) * (child.alpha / child.lora_rank)
                    child.base_layer.weight.add_(lora_adjustment)
                
                # 替换回原来的 base_layer
                setattr(module, name, child.base_layer)
            else:
                search_lora_linear(child, new_prefix)

    search_lora_linear(module, [])
    # 解冻原模型
    for name, param in module.named_parameters():
        param.requires_grad = True

# 合并
merge_lora(model)